In [1]:
from PySide6.QtWidgets import QApplication, QWidget, QPushButton
import subprocess
import sys
import os
from PySide6.QtWidgets import (
    QApplication, QWidget, QLabel, QHBoxLayout, QMenu,
    QSystemTrayIcon, QStyle, QPushButton, QListWidget,
    QListWidgetItem, QVBoxLayout, QStackedWidget, QLineEdit,
    QLayout, QFormLayout, QPlainTextEdit, QComboBox, QDialog, QFileDialog, QDialogButtonBox,
    QDateTimeEdit ,QSpinBox, QCheckBox
)
from PySide6.QtCore import Qt, QTimer, QPoint, QRectF, Signal, QThread, QPropertyAnimation, QEasingCurve, QBuffer, QRect, QProcess
from PySide6.QtGui import (
    QPainter, QColor, QAction, QPixmap, QGuiApplication,
    QIcon, QBrush, QPen, QFont, QImage, QRegion, 
)
from typing import Optional

class complex_config_page(QDialog):
    def __init__(self,parent=None , dict:Optional[dict] = None):
        super().__init__(parent)
        self.setWindowTitle("任务配置")
        self.resize(450, 300)
        self.main_layout = QVBoxLayout()
        self.Form = QFormLayout()
        self.main_layout.addLayout(self.Form)
        self.dict = dict if dict else {}
        self.setup_ui()

        if dict:
            self.load_config_to_ui()
        
        self.button_box = QDialogButtonBox(QDialogButtonBox.Ok | QDialogButtonBox.Cancel)
        self.button_box.accepted.connect(self.accept) # 点击OK触发 accept()
        self.button_box.rejected.connect(self.reject) # 点击Cancel触发 reject()
        self.main_layout.addWidget(self.button_box)
        self.setLayout(self.main_layout)
        self.show()
    def setup_ui(self):
        self.task_path, self.task_cmd, self.cmd_args = QLineEdit(), QLineEdit(), QLineEdit()
        
        self.task_cmd.setObjectName("task_cmd")
        self.task_cmd.setPlaceholderText("请输入程序路径，必选")
        self.task_cmd_btn = QPushButton("选择程序")
        self.task_cmd_btn.clicked.connect(self.task_cmd_select)
        self.task_cmd_layout = QHBoxLayout()
        self.task_cmd_layout.addWidget(self.task_cmd)
        self.task_cmd_layout.addWidget(self.task_cmd_btn)
        self.cmd_args.setObjectName("cmd_args")
        self.cmd_args.setPlaceholderText("请输入程序参数，可留空")
        self.task_path.setObjectName("task_path")
        self.task_path.setPlaceholderText("请输入程序运行目录，可留空")
        self.task_path_btn = QPushButton("选择目录")
        self.task_path_btn.clicked.connect(self.task_path_select)
        self.task_path_layout = QHBoxLayout()
        self.task_path_layout.addWidget(self.task_path)
        self.task_path_layout.addWidget(self.task_path_btn)
        
        self.max_exec_time = QSpinBox()
        self.max_exec_time.setRange(0, 3600)
        self.max_exec_time.setSuffix("秒")
        self.max_exec_time.setSpecialValueText("无限制")
        self.max_exec_time.setToolTip("设置最大运行时间（秒）。输入 0 表示不限制时间。")
        self.start_up = QCheckBox("开机启动")
        self.start_up.setToolTip("设置任务是否开机启动")
        self.auto_restart = QCheckBox("自动重启")
        self.auto_restart.setToolTip("当达到任务最大运行时间，或进程退出后，是否自动重启任务")
        self.start_time_hour = QSpinBox()
        self.start_time_hour.setRange(0, 23)
        self.start_time_hour.setSuffix("时")
        self.start_time_minute = QSpinBox()
        self.start_time_minute.setRange(0, 59)
        self.start_time_minute.setSuffix("分")
        self.enable_timer = QCheckBox("定时启动")
        self.enable_timer.setToolTip("设置任务是否定时启动")
        self.set_time_layout = QHBoxLayout()
        self.set_time_layout.addWidget(QLabel("定时启动（每日）"))
        self.set_time_layout.addWidget(self.start_time_hour)
        self.set_time_layout.addWidget(self.start_time_minute)
        self.set_time_layout.addWidget(self.enable_timer)
        self.set_time_layout.addWidget(self.start_up)
        self.set_time_layout.addWidget(self.auto_restart)

        self.main_layout.addLayout(self.set_time_layout)
        self.Form.addRow("运行时间",self.max_exec_time)
        self.Form.addRow("运行路径",self.task_path_layout)
        self.Form.addRow("运行程序",self.task_cmd_layout)
        self.Form.addRow("运行参数",self.cmd_args)

    def task_path_select(self):
        file_path = QFileDialog.getExistingDirectory(self, "选择目录",)
        if file_path:
            self.task_path.setText(file_path)
    def task_cmd_select(self):
        file_name,_ = QFileDialog.getOpenFileName(self, "选择程序",)
        if file_name:
            self.task_cmd.setText(file_name)
    def load_config_to_ui(self):
        self.task_cmd.setText(self.dict["task_cmd"])
        self.task_path.setText(self.dict["task_path"])
        self.cmd_args.setText(self.dict["cmd_args"])
        self.start_up.setChecked(self.dict["start_up"])
        self.auto_restart.setChecked(self.dict["auto_restart"])
        self.max_exec_time.setValue(self.dict["max_exec_time"])
        self.start_time_hour.setValue(self.dict["start_time_hour"])
        self.start_time_minute.setValue(self.dict["start_time_minute"])
        self.enable_timer.setChecked(self.dict["enable_timer"])

    def accept(self) -> None:
        self.dict = {
            "task_cmd": self.task_cmd.text(),
            "task_path": self.task_path.text(),
            "cmd_args": self.cmd_args.text(),
            "start_up": self.start_up.isChecked(),
            "auto_restart": self.auto_restart.isChecked(),
            "max_exec_time": self.max_exec_time.value(),
            "start_time_hour": self.start_time_hour.value(),
            "start_time_minute": self.start_time_minute.value(),
            "enable_timer": self.enable_timer.isChecked(),

        }
        return super().accept()


app = QApplication(sys.argv)

dict_config = eval("{'task_cmd': 'D:/code/python/uni_mngr/test.py', 'task_path': 'D:/code/python/uni_mngr', 'cmd_args': '', 'start_up': True, 'auto_restart': True, 'max_exec_time': 0, 'start_time_hour': 7, 'start_time_minute': 3, 'enable_timer': True}")
window = complex_config_page(dict=dict_config)
result = window.exec()
if result == QDialog.Accepted:
    print(window.dict)